In [1]:
print("12")

12


In [2]:
!uv pip install langchain openai tiktoken rapidocr-onnxruntime python-dotenv langchain-community 

Using Python 3.11.9 environment at: C:\Users\Friends shop\OneDrive\Desktop\LLMOPS_SERIES\.venv
Audited 6 packages in 6.06s


In [3]:
import os

from dotenv import load_dotenv

load_dotenv()


os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

In [4]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader("../data/agentic.txt")  # .. matlab ek folder upar jao
documents = loader.load()
print(documents)


[Document(metadata={'source': '../data/agentic.txt'}, page_content="Understanding Agentic AI\n\nAgentic AI refers to a new paradigm in artificial intelligence where systems are designed not just to respond to queries or perform specific tasks, but to operate autonomously towards achieving predefined goals. This involves capabilities such as planning, reasoning, executing actions, and continuously adapting to dynamic environments without constant human intervention.\n\nKey Characteristics of Agentic AI\n\nAgentic AI systems are distinct from traditional AI models due to several core characteristics:\n\nGoal-Oriented: They possess a clear objective or goal that they strive to achieve.\n\nAutonomy: They can operate independently, making decisions and taking actions without continuous human oversight.\n\nPerception: They can interpret information from their environment to inform their decisions.\n\nPlanning and Reasoning: They can formulate plans to reach their goals and reason about the c

In [5]:
documents[0].page_content[:500]

'Understanding Agentic AI\n\nAgentic AI refers to a new paradigm in artificial intelligence where systems are designed not just to respond to queries or perform specific tasks, but to operate autonomously towards achieving predefined goals. This involves capabilities such as planning, reasoning, executing actions, and continuously adapting to dynamic environments without constant human intervention.\n\nKey Characteristics of Agentic AI\n\nAgentic AI systems are distinct from traditional AI models due t'

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter


In [7]:
text_splitter=RecursiveCharacterTextSplitter(chunk_size=100,chunk_overlap=20)

In [8]:
text_chunks=text_splitter.split_documents(documents)

In [9]:
text_chunks

[Document(metadata={'source': '../data/agentic.txt'}, page_content='Understanding Agentic AI'),
 Document(metadata={'source': '../data/agentic.txt'}, page_content='Agentic AI refers to a new paradigm in artificial intelligence where systems are designed not just'),
 Document(metadata={'source': '../data/agentic.txt'}, page_content='designed not just to respond to queries or perform specific tasks, but to operate autonomously'),
 Document(metadata={'source': '../data/agentic.txt'}, page_content='autonomously towards achieving predefined goals. This involves capabilities such as planning,'),
 Document(metadata={'source': '../data/agentic.txt'}, page_content='such as planning, reasoning, executing actions, and continuously adapting to dynamic environments'),
 Document(metadata={'source': '../data/agentic.txt'}, page_content='environments without constant human intervention.'),
 Document(metadata={'source': '../data/agentic.txt'}, page_content='Key Characteristics of Agentic AI'),
 Documen

In [10]:
uv add faiss-cpu

Note: you may need to restart the kernel to use updated packages.


c:\Users\Friends shop\OneDrive\Desktop\LLMOPS_SERIES\.venv\Scripts\python.exe: No module named uv


In [11]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS


In [12]:
embedings=OpenAIEmbeddings()

In [16]:
vectorstore=FAISS.from_documents(text_chunks,embedings)

In [17]:
retriever=vectorstore.as_retriever()

In [20]:
# Perform similarity search
query = " Agentic AI?"
docs = vectorstore.similarity_search(query, k=4)

# Display the results
for i, doc in enumerate(docs):
    print(f"Document {i+1}:")
    print(doc.page_content)
    print("-" * 50)

Document 1:
Understanding Agentic AI
--------------------------------------------------
Document 2:
How Agentic AI Works
--------------------------------------------------
Document 3:
The Future of Agentic AI
--------------------------------------------------
Document 4:
Key Characteristics of Agentic AI
--------------------------------------------------


In [24]:
from langchain_core.prompts import ChatPromptTemplate

template = """You are an assistant for question-answering tasks.
Use the following pieces of retrieved context to answer the question.
If you don't know the answer, just say that you don't know.

Context: {context}

Question: {question}

Answer:"""

prompt = ChatPromptTemplate.from_template(template)


In [25]:
prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks.\nUse the following pieces of retrieved context to answer the question.\nIf you don't know the answer, just say that you don't know.\n\nContext: {context}\n\nQuestion: {question}\n\nAnswer:"), additional_kwargs={})])

In [27]:
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

output_parser = StrOutputParser()
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)


In [32]:
from langchain_core.runnables import RunnablePassthrough

rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | output_parser
)

rag_chain.invoke("What is the capital of France?")


"I don't know."